# SX: Pressure distributions and regression diagnostics

Objective: compare the 0-10 kbar pressure distributions of the cleaned independent experimental dataset and the unique training datasets represented in the pressure-model pool, then quantify how each model's linear pressure-error trend changes with true pressure.

Analysis contract:
- Histogram range: measured pressure from 0 to 10 kbar, inclusive.
- Bin edges: 0, 1, ..., 10 kbar (1-kbar bin width; the 10-kbar endpoint is included in the last bin).
- One histogram figure: Independent + seven training datasets, deduplicated by publication/data source.
- Each panel title reports the histogram sample count $n$ without thousands separators.
- Mean uses only the displayed 0-10 kbar interval and is reported to one decimal place.
- Training source: `X_cpx_training` for every retained model.
- Residual definition: $\Delta P=P_{pred}-P_{true}$. OLS error regressions use every cleaned independent sample with a finite prediction and $0\leq P_{true}\leq10$ kbar.
- Each model is shown as one OLS line representing the fitted linear mean trend in $\Delta P$; no residual-shading envelope is drawn.
- All residual axes use a shared range and include a prominent $\Delta P=0$ reference line.
- Model-line meanings are defined once in a shared legend centered below the complete figure.

In [ ]:
from pathlib import Path
import importlib
import sys


def find_project_root(start=None):
    """Find the repository root from the active notebook directory."""
    path = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (path, *path.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find pyproject.toml above the current directory.")


PROJECT_ROOT = find_project_root()
PAPER_DIR = PROJECT_ROOT / "paper"
DATA_DIR = PAPER_DIR / "data"
CACHE_DIR = PAPER_DIR / ".cache"
IMAGES_DIR = PAPER_DIR / ".images"
for directory in (CACHE_DIR, IMAGES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from aims4pt.model_tools.model_registry import ALL_MODELS_MODULES, get_models_initial_pools
from paper.scripts.sx_pressure_histograms_and_regression import (
    CPX_COLUMNS,
    LIQ_COLUMNS,
    PRESSURE_COLUMN,
    TEMPERATURE_COLUMN,
    calculate_regression_statistics,
    collect_pressure_distributions,
    export_summary_tables,
    load_or_calculate_test_predictions,
    plot_pressure_histograms,
    plot_regression_lines,
    prepare_independent_dataset,
    save_figure_bundle,
    select_unique_training_models,
)

## 1. Reproduce the independent-data filtering and fixed split

In [ ]:
independent_path = DATA_DIR / "independent_data_raw.xlsx"
independent_df, train_index, test_index, filtering_audit = prepare_independent_dataset(
    independent_path,
    test_size=0.2,
    random_state=42,
)

independent_cpx = independent_df.loc[:, CPX_COLUMNS].copy()
independent_liq = independent_df.loc[:, LIQ_COLUMNS].copy()
independent_pressure = independent_df.loc[:, PRESSURE_COLUMN].copy()
test_cpx = independent_df.loc[test_index, CPX_COLUMNS].copy()
test_liq = independent_df.loc[test_index, LIQ_COLUMNS].copy()
test_pressure = independent_df.loc[test_index, PRESSURE_COLUMN].copy()

display(filtering_audit)
print(f"Independent cleaned N = {len(independent_df)}")
print(
    "Independent N within 0-10 kbar =",
    int(independent_pressure.between(0, 10, inclusive="both").sum()),
)
print(
    "Independent N below 1 kbar now included in the histogram =",
    int(independent_pressure.between(0, 1, inclusive="left").sum()),
)
print(f"Fixed test-subset N = {len(test_index)}")
print(
    "Fixed test-subset N within 1-10 kbar =",
    int(test_pressure.between(1, 10, inclusive="both").sum()),
)

## 2. Load all pressure models and their attached training datasets

In [ ]:
for module_name in ALL_MODELS_MODULES:
    importlib.import_module(module_name)

pressure_models = get_models_initial_pools("P", "both", False)
training_models = select_unique_training_models(pressure_models)

print(f"Pressure models used for regression: {len(pressure_models)}")
print(f"Unique training datasets used for histograms: {len(training_models)}")
for model in training_models:
    print(f"- {model.model_name}")

## 3. Histograms, training-dataset averages, and linear error regressions

Training datasets shared by cpx-only and cpx-liquid models from the same study are shown once. The single 2×4 figure contains the Independent dataset plus Chi23, Jor22, AgL24, Pet20, Hig21, Wan21, and NP17; each title includes $n$ without thousands separators. The dark-gray dash-dot line is the mean of the displayed 0-10 kbar interval; its annotation omits the range subscript and reports one decimal place. Twin right axes show OLS regression lines for $\Delta P=P_{pred}-P_{true}$ against $P_{true}$, fitted to all cleaned independent data at $0\leq P_{true}\leq10$ kbar. Only the fitted lines are drawn, without a residual-shading envelope. A single untitled shared legend is centered below the complete figure, and the $\Delta P=0$ line is emphasized. Study models are placed in their corresponding training-data panel. Putirka equations without a training-data panel are excluded from this histogram figure.

In [ ]:
pressure_datasets, histogram_summary = collect_pressure_distributions(
    independent_df[PRESSURE_COLUMN],
    training_models,
    p_min=0,
    p_max=10,
)

In [ ]:
test_prediction_cache = CACHE_DIR / "predicted_P_unseen_testing_set.csv"
test_predictions = load_or_calculate_test_predictions(
    test_prediction_cache,
    pressure_models,
    test_cpx,
    test_liq,
)
independent_prediction_cache = CACHE_DIR / "predicted_P_all_cleaned_independent.csv"
independent_predictions = load_or_calculate_test_predictions(
    independent_prediction_cache,
    pressure_models,
    independent_cpx,
    independent_liq,
)

# Keep the separate y_pred-vs-y diagnostics on the fixed held-out subset.
regression_summary = calculate_regression_statistics(
    test_pressure,
    test_predictions,
    pressure_models,
    p_min=1,
    p_max=10,
)

# Fit linear error trends to all cleaned independent data from 0 to 10 kbar.
error_regression_summary = calculate_regression_statistics(
    independent_pressure,
    independent_predictions,
    pressure_models,
    p_min=0,
    p_max=10,
)

display(
    histogram_summary.style.format(
        {
            "fraction_0_10_kbar": "{:.3f}",
            "mean_P_0_10_kbar": "{:.2f}",
            "median_P_0_10_kbar": "{:.2f}",
        },
        na_rep="NA",
    )
)

histogram_fig, histogram_axes = plot_pressure_histograms(
    pressure_datasets,
    regression_summary=error_regression_summary,
    p_min=0,
    p_max=10,
    ncols=4,
    delta_p_ylim=(-8, 8),
)
histogram_figure_paths = save_figure_bundle(
    histogram_fig,
    IMAGES_DIR,
    "Fig_SX_pressure_histograms_unique_datasets_0_10_kbar",
)
plt.show()
histogram_figure_paths

## 4. Held-out OLS slopes and 1:1-line intersections

The separate slope diagnostic retains the original OLS fits $\hat{P}=aP+b$ and $\Delta P=cP+d$ on the fixed held-out subset with finite predictions and measured pressure from 1 to 10 kbar. Its table reports both regressions, residual standard deviation, $R^2$, the formal intersection with the 1:1 line, average measured/predicted pressure, bias, and RMSE. These held-out estimates are separate from the all-independent-data error regressions drawn on the histogram figure.

In [ ]:
display(
    regression_summary.style.format(
        {
            "slope": "{:.3f}",
            "intercept_kbar": "{:.2f}",
            "deltaP_slope": "{:.3f}",
            "deltaP_intercept_kbar": "{:.2f}",
            "deltaP_sigma_kbar": "{:.2f}",
            "R_squared": "{:.3f}",
            "intersection_with_1to1_kbar": "{:.2f}",
            "mean_y_true_kbar": "{:.2f}",
            "mean_y_pred_kbar": "{:.2f}",
            "bias_kbar": "{:+.2f}",
            "RMSE_kbar": "{:.2f}",
        },
        na_rep="NA",
    )
)

In [ ]:
regression_fig, regression_axes = plot_regression_lines(
    regression_summary,
    p_min=1,
    p_max=10,
)
regression_figure_paths = save_figure_bundle(
    regression_fig,
    IMAGES_DIR,
    "Fig_SX_pressure_test_regression_lines_1_10_kbar",
)
plt.show()
regression_figure_paths

## 5. Export source tables and compact review table

In [ ]:
histogram_csv, regression_csv = export_summary_tables(
    histogram_summary,
    regression_summary,
    CACHE_DIR,
)
error_regression_csv = CACHE_DIR / "SX_pressure_error_regression_all_independent_0_10.csv"
error_regression_summary.to_csv(error_regression_csv, index=False)

compact_columns = [
    "model_short",
    "phase_type",
    "N",
    "slope",
    "intersection_with_1to1_kbar",
    "intersection_note",
    "mean_y_pred_kbar",
]
display(
    regression_summary[compact_columns].style.format(
        {
            "slope": "{:.3f}",
            "intersection_with_1to1_kbar": "{:.2f}",
            "mean_y_pred_kbar": "{:.2f}",
        },
        na_rep="NA",
    )
)
display(
    error_regression_summary.loc[
        ~error_regression_summary["model_short"].str.startswith("Pu08_"),
        [
            "model_short",
            "phase_type",
            "N",
            "deltaP_slope",
            "deltaP_intercept_kbar",
            "deltaP_sigma_kbar",
            "bias_kbar",
            "RMSE_kbar",
        ],
    ].style.format(
        {
            "deltaP_slope": "{:.3f}",
            "deltaP_intercept_kbar": "{:.2f}",
            "deltaP_sigma_kbar": "{:.2f}",
            "bias_kbar": "{:+.2f}",
            "RMSE_kbar": "{:.2f}",
        }
    )
)
print("Histogram summary:", histogram_csv)
print("Regression summary (fixed test subset):", regression_csv)
print("Error regressions (all independent, 0 <= P_true <= 10):", error_regression_csv)